# 03 — Log preprocessing

Load CSVs under `data/processed/logs_raw/`, apply **global** categorical encodings (same integer meaning across buildings), sort by time, and write processed files plus `encoding_metadata.json` to `data/processed/logs_processed/`.

**By default** only buildings listed in `src.config.BUILDINGS` (same five as the sensor LSTM study) are processed. Pass `process_all_buildings=True` to `run_preprocessing_pipeline` when you scale to the full log corpus.

If you previously processed hundreds of buildings, delete stale `*_processed.csv` files you do not need, or use `process_all_buildings=True` only when you intend to refresh everything.

Implementation lives in `scripts/log_pipeline/preprocessing.py` (this notebook only wires paths and options).

In [9]:
import os
import sys

import pandas as pd

# Notebook cwd: notebooks/
SCRIPTS = os.path.abspath(os.path.join(os.getcwd(), "..", "scripts"))
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

from log_pipeline.preprocessing import (
    LSTM_FEATURE_COLUMNS,
    run_preprocessing_pipeline,
)

LOGS_RAW = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "processed", "logs_raw"))
LOGS_PROCESSED = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "processed", "logs_processed"))

os.makedirs(LOGS_PROCESSED, exist_ok=True)
print("logs_raw:", LOGS_RAW)
print("logs_processed:", LOGS_PROCESSED)
print("Default LSTM feature columns:", LSTM_FEATURE_COLUMNS)

logs_raw: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw
logs_processed: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_processed
Default LSTM feature columns: ['event_type', 'event_code', 'severity', 'device_id', 'subsystem', 'building_id', 'value']


In [11]:
# Set encode_message=True to add a global message_id column (then add "message_id" to feature_cols in notebook 04).
# process_all_buildings=False (default): only src.config.BUILDINGS. Use True to preprocess every CSV in logs_raw.
summary = run_preprocessing_pipeline(
    LOGS_RAW,
    LOGS_PROCESSED,
    encode_message=False,
    process_all_buildings=False,
)

for k, v in summary.items():
    print(f"{k}: {v}")

files_processed: 5
processed_paths: ['D:\\Uni\\Sem 4\\PBL 1\\smart-building-anomaly-detection\\data\\processed\\logs_processed\\Office_Annika_processed.csv', 'D:\\Uni\\Sem 4\\PBL 1\\smart-building-anomaly-detection\\data\\processed\\logs_processed\\Office_Cristina_processed.csv', 'D:\\Uni\\Sem 4\\PBL 1\\smart-building-anomaly-detection\\data\\processed\\logs_processed\\Office_Jesus_processed.csv', 'D:\\Uni\\Sem 4\\PBL 1\\smart-building-anomaly-detection\\data\\processed\\logs_processed\\PrimClass_Jaylin_processed.csv', 'D:\\Uni\\Sem 4\\PBL 1\\smart-building-anomaly-detection\\data\\processed\\logs_processed\\PrimClass_Jolie_processed.csv']
metadata_path: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_processed\encoding_metadata.json
category_vocab_sizes: {'event_type': 4, 'event_code': 8, 'device_id': 3, 'subsystem': 1, 'building_id': 5}
lstm_feature_columns_default: ['event_type', 'event_code', 'severity', 'device_id', 'subsystem', 'building_id', 'value']


In [13]:
processed = [f for f in os.listdir(LOGS_PROCESSED) if f.endswith("_processed.csv")]
if not processed:
    print("No processed CSVs yet — run log simulation (02) first or check LOGS_RAW.")
else:
    sample = os.path.join(LOGS_PROCESSED, sorted(processed)[0])
    display(pd.read_csv(sample).head())
    print("Shape:", pd.read_csv(sample).shape)

,timestamp,building_id,device_id,subsystem,event_type,event_code,severity,value,message,anomaly_link
0,2015-01-08 23:00:00,0,0,0,3,1,2,25.781216,DEVICE_OFFLINE occurred,0
1,2015-01-09 00:00:00,0,0,0,0,7,0,28.073766,SETPOINT_CHANGE occurred,0
2,2015-01-09 00:00:00,0,1,0,1,6,2,26.954752,OVERLOAD occurred,0
3,2015-01-09 00:55:00,0,0,0,2,0,1,26.801253,CALIBRATION_WARNING occurred,1
4,2015-01-09 01:00:00,0,0,0,2,2,1,25.192031,DRIFT_DETECTED occurred,1


Shape: (8688, 10)
